In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from itertools import product

import sys

from pathlib import Path
nb_dir = Path.cwd()
help_func_path = os.path.join(nb_dir,'helper_code')

sys.path.append(help_func_path) 

from plotting_functions import scatter_helper, plot_3d_scatters
from saving_functions import save_dataset,delete_dataset

#### uses sklearn.datasets.make_blobs which samples from a GMM under the hood

In [ ]:
from sklearn import datasets

## homoskedastic, balanced clusters

In [ ]:
X,y = datasets.make_blobs(n_samples=10000,n_features=1000, centers=40,cluster_std=0.5)
plot_3d_scatters(X,'',y=y)

In [ ]:
# delete_dataset('GMM_isotropic_homoskedastic_balanced','../all_data/')

In [ ]:
data_gen_name = 'GMM_isotropic_homoskedastic_balanced'
category_lst = ['blobbiness']

random_seeds_lst = [42,980]

n_samples_lst = [1000,10000]
n_features_lst = [10,100,1000,10000]
# define some number of clusters according to the number of features
# {num_feature : [list of num_cluster params to sweep over]}
num_clusters_dict = {10 : [3,5,7], 100 : [3,10], 1000 : [3,10,40], 10000 : [3,10,40]}
cluster_std_lst = [0.2, 0.5,1, 2]

for n_samples, n_features in product(n_samples_lst, n_features_lst):
    
    print(n_samples, n_features, X.shape)

    for n_centers in num_clusters_dict[n_features]:

        for cluster_std in cluster_std_lst:

            for sample_num,seed in enumerate(random_seeds_lst):
                
                X, y = datasets.make_blobs(n_samples=n_samples,n_features=n_features, random_state=seed, centers=n_centers,cluster_std=cluster_std)
                
                params_dict = {
                    'num_clusters': n_centers,
                    'cluster_std' : cluster_std
                }
                
                extra_mtx_params = {
                    'cluster_labels' : y
                }
                
                try:
                    save_dataset(X, 
                                generator=data_gen_name,
                                params=params_dict,
                                seed=seed,
                                category=category_lst,
                                extra_arrays=extra_mtx_params, 
                                root='../all_data/')

                except:
                    print('error in saving')

In [ ]:
m = pd.read_csv("../all_data/manifest.csv")
generator = data_gen_name
params_lst = [f'params{i}' for i in range(0,20) if i%8==2]
m = m[(m.generator == generator) & (m.seed==42) & (m.params_id.isin(params_lst))]
m

In [ ]:
max_pts = 500
for r in m.groupby(["generator", "params_id"]).head(1).itertuples():
    X = np.load(Path('../all_data/')  / f"{r.dataset_id}.npy")
    if len(X) > max_pts:
        rng = np.random.default_rng(0)       
        X = X[rng.choice(len(X), max_pts, replace=False)]
    # plot_3d_scatters(X, f"{r.generator}/{r.params_id}, N={r.n_rows},P={r.n_cols}")
    plot_3d_scatters(X, f"{r.generator}/{r.params_id}, N={r.n_rows},P={r.n_cols}",custom_inx=np.random.randint(0,r.n_cols,3))

## homoskedastic, imbalanced clusters

$$w_k = \dfrac{1}{k^s}, k = 1,2,3,..K$$

w_k = cluster size proportion for cluster k, K = total number of clusters, s = hyperparameter to vary

s = 0 is when w_k = 1 (balanced case)

normalize w_k's to get proportions

as s increases, w_0 >> w_1 >> w_2 >> ..

a power law reln. (ref https://en.wikipedia.org/wiki/Power_law)

In [ ]:
s = 0
n_centers = 3

def get_cluster_sizes_power_law(decay_exp, n_clusters, n_samples, min_count=20):
    """
    get decaying cluster imbalance
    currently relies on manual tuning of the decay parameter
    default: apply a min count per cluster of 20
    """
    if n_samples < min_count * n_clusters:
        raise ValueError(f"n_samples={n_samples} cannot give {n_clusters} clusters "
                         f"at least {min_count} points each")
        
    w = 1 / np.arange(1, n_clusters + 1) ** decay_exp
    w /= w.sum()
    spare = n_samples - min_count * n_clusters
    counts = min_count + np.floor(w * spare).astype(int)
    counts[0] += n_samples - counts.sum()
    return counts.tolist()


In [ ]:
# get_cluster_sizes_power_law(0.5,40,1000)
# get_cluster_sizes_power_law(4,3,100)

In [ ]:

# {num_cluster : [list of decay parameters]}
size_decay_dict = {
    3 : [0.5,1,2],
    10 : [0.5,1],
    40 : [0.5]
}

for nc in [3,10,40]:
    
    s_lst = size_decay_dict[nc]
            
    for s in s_lst:
        counts = get_cluster_sizes_power_law(s,nc,10000)
        print(max(counts)/min(counts))
        print(counts)
        print('---')

In [ ]:
# delete_dataset('GMM_isotropic_homoskedastic_imbalanced','../all_data/')

In [ ]:
data_gen_name = 'GMM_isotropic_homoskedastic_imbalanced'
category_lst = ['blobbiness']

random_seeds_lst = [42,980]

n_samples_lst = [1000,10000]
n_features_lst = [10,100,1000,10000]
# define some number of clusters according to the number of features
# {num_feature : [list of num_cluster params to sweep over]}
num_clusters_dict = {
    10 : [3,5,7], 100 : [3,10], 
    1000 : [3,10,40], 10000 : [3,10,40]
}

# {num_cluster : [list of decay parameters]}
size_decay_dict = {
    3 : [0.5,2], 5 : [0.5,2], 7 : [0.5,2],
    10 : [0.5], 40 : [0.5]
}

# MORE EXTENDED, IF REQUIRED:
# size_decay_dict = {
#     3 : [0.5,1,2], 5 : [0.5,1,2], 7 : [0.5,1,2],
#     10 : [0.5,1], 40 : [0.5]
# }

cluster_std_lst = [0.2, 0.5,1, 2]


for n_samples, n_features in product(n_samples_lst, n_features_lst):
    
    print(n_samples, n_features, X.shape)

    for n_centers in num_clusters_dict[n_features]:
        
        for decay_exp in size_decay_dict[n_centers]:
            
            cluster_sizes = get_cluster_sizes_power_law(decay_exp,n_centers,n_samples)

            for cluster_std in cluster_std_lst:

                for sample_num,seed in enumerate(random_seeds_lst):
                    
                    X, y = datasets.make_blobs(n_samples=cluster_sizes,n_features=n_features, random_state=seed, centers=None,cluster_std=cluster_std)
                    
                    params_dict = {
                        'num_clusters': n_centers,
                        'cluster_std' : cluster_std,
                        'cluster_size_decay_param' : decay_exp,
                        'n_samples' : n_samples,
                        'cluster_size_method' : 'use the get_cluster_sizes_power_law method described in the data gen notebook'
                    }
                    
                    extra_mtx_params = {
                        'cluster_labels' : y,
                        'cluster_sizes' : cluster_sizes
                    }
                    
                    try:
                        save_dataset(X, 
                                    generator=data_gen_name,
                                    params=params_dict,
                                    seed=seed,
                                    category=category_lst,
                                    extra_arrays=extra_mtx_params, 
                                    root='../all_data/')

                    except:
                        print('error in saving')

In [ ]:
m = pd.read_csv("../all_data/manifest.csv")

# some manifest.csv queries
# m.groupby(['n_rows','n_cols'])[['generator']].value_counts()
# m.groupby(['n_rows','n_cols'])['dataset_id'].count()

In [ ]:
generator = data_gen_name
params_lst = [f'params{i}' for i in range(0,100) if i%8==2]
m = m[(m.generator == generator) & (m.seed==42) & (m.params_id.isin(params_lst))]
m

In [ ]:
max_pts = 1000
for r in m.groupby(["generator", "params_id"]).head(1).itertuples():
    X = np.load(Path('../all_data/')  / f"{r.dataset_id}.npy")
    if len(X) > max_pts:
        rng = np.random.default_rng(0)       
        X = X[rng.choice(len(X), max_pts, replace=False)]
    # plot_3d_scatters(X, f"{r.generator}/{r.params_id}, N={r.n_rows},P={r.n_cols}")
    plot_3d_scatters(X, f"{r.generator}/{r.params_id}, N={r.n_rows},P={r.n_cols}",custom_inx=np.random.randint(0,r.n_cols,3))

In [ ]:
# old function, had the left_sample_prop as well but that kinda adds an unnecessary parameter
# def get_cluster_sizes_power_law(exp_decay,n_clusters,n_samples,min_count_per_cluster=10,left_sample_prop=0.5):    
    
#     """
#     get exponentially decaying cluster imbalance
#     currently relies on manual tuning of the decay parameter for 
#     default: apply a min count per cluster of 10 if doing that preserves at least 50% of samples
#     """
    
#     w = 1 / np.arange(1, n_clusters + 1) ** exp_decay  
#     norm_w = w / w.sum()
#     print(norm_w)
    
#     min_counts = np.full(n_clusters, min_count_per_cluster)
#     left_samples = n_samples - min_count_per_cluster*n_clusters        
    
#     if left_samples/n_samples > left_sample_prop:
#         counts = np.floor(norm_w * left_samples).astype(int) + min_counts
        
#     else:
#         counts = np.floor(norm_w * n_samples).astype(int)
        
#     counts[0] += n_samples - counts.sum()    
    
#     if counts.min() < 10:
#         print('min counts unreliable at this decay rate')
    
#     return counts
